In [5]:
import pandas as pd
import numpy as np

import yfinance as yf
import datetime as dt

In [6]:
TICKERS_FILE: str = 'us-tickers.txt'
DATA_COLS: list[str] = ['Open', 'High', 'Low', 'Close', 'Volume']

START_DATE: str = '2019-01-01'
END_DATE: str = '2026-05-01'

In [7]:
def get_tickers_data(tickers_path: str = TICKERS_FILE, cols: list[str] = DATA_COLS, start: str = START_DATE, end: str = END_DATE) -> pd.DataFrame:
    start_date, end_date = dt.datetime.strptime(start, "%Y-%m-%d"), dt.datetime.strptime(end, "%Y-%m-%d")

    with open(tickers_path, 'r') as f:
        tickers = [line.strip() for line in f if line.strip()]

    dfs = []
    for ticker in tickers:
        try:
            data = yf.Ticker(ticker).history(start=start_date, end=end_date)
            if data.empty:
                continue

            df = data.reset_index()
            df['Date'] = df['Date'].dt.tz_localize(None)
            df['Ticker'] = ticker
            df = df[['Date', 'Ticker'] + cols]

            dfs.append(df)

        except Exception:
            continue

    # combine all tickers
    full_df = pd.concat(dfs, ignore_index=True)

    full_df = full_df.set_index(['Date', 'Ticker']).sort_index()

    return full_df.dropna()

In [10]:
df = pd.read_parquet('data.parquet')

In [11]:
df

Open        High         Low       Close    Volume
Date       Ticker                                                          
2019-01-02 ABEO    175.000000  179.500000  170.250000  175.750000     16636
           ABEV      3.018187    3.199278    2.995551    3.169096  20573600
           ABG      65.709999   68.459999   65.709999   68.120003    252900
           ABM      27.181189   27.284702   26.447960   26.982786   1039300
           ABR       4.645739    4.734007    4.617864    4.729362    715900
...                       ...         ...         ...         ...       ...
2026-04-30 ZVIA      1.250000    1.290000    1.220000    1.280000    214000
           ZVRA      9.770000   10.310000    9.750000   10.170000   1062200
           ZWS      51.169998   52.340000   50.990002   51.959999   1156800
           ZYBT      0.850000    0.944000    0.830000    0.920000     61800
           ZYME     27.975000   28.184999   27.180000   27.540001    482700

[8888812 rows x 5 columns]